In [ ]:
import numpy as np

# Load the data from the .npy file
data = np.load(
    '/home/juma/code/StrongSORT/results/StrongSORT_Git/MOT17_val_YOLOX+BoT/MOT17-02-FRCNN.npy', allow_pickle=True)

# Check the type of the loaded data
print("Type of the loaded data:", type(data))

# Check the shape of the data (if it's a numpy array)
if isinstance(data, np.ndarray):
    print("Shape of the loaded numpy array:", data.shape)

# If the loaded data is a dictionary, print its keys
elif isinstance(data, dict):
    print("Keys in the loaded dictionary:", data.keys())

# Display the first few items (for visualization purposes)
print("\nSample data:")
print(data[:10] if isinstance(data, np.ndarray) else {
      key: data[key] for key in list(data.keys())[:10]})

In [ ]:
data[0, :10]


In [ ]:
import numpy as np


def parse_detections(detection_mat, frame_idx, min_height=0):
    """Parse detections for given frame index from the raw detection matrix.

    Parameters
    ----------
    detection_mat : ndarray
        Matrix of detections. The first 10 columns of the detection matrix are
        in the standard MOTChallenge detection format. The next 128 columns are apperance features
    frame_idx : int
        The frame index.
    min_height : Optional[int]
        A minimum detection bounding box height. Detections that are smaller
        than this value are disregarded.

    Returns
    -------
    List[tuple]
        Returns parsed detections at the given frame index as a list of tuples.

    """
    frame_indices = detection_mat[:, 0].astype(int)
    mask = frame_indices == frame_idx

    parsed_detections = []
    for row in detection_mat[mask]:
        bbox, confidence = row[2:6], row[6]
        if bbox[3] < min_height:
            continue
        parsed_detections.append((bbox, confidence))
    return parsed_detections


# Example usage
data = np.load(
    '/home/juma/code/StrongSORT/results/StrongSORT_Git/MOT17_val_YOLOX+simpleCNN/MOT17-02-FRCNN.npy', allow_pickle=True)
frame_detections = parse_detections(data, frame_idx=302)
print(frame_detections)

In [ ]:
data.shape


In [ ]:
data[0, 10:]


# Minimalsitic REID code



In [ ]:
from __future__ import division, print_function, absolute_import
from ultralytics import YOLO

import os
import time

import cv2
import numpy as np

from application_util import preprocessing
from application_util import visualization
from deep_sort import nn_matching
from deep_sort.detection import Detection
from deep_sort.tracker import Tracker
from opts import opt

In [ ]:
import cv2
from fastreid.config import get_cfg
from fastreid.engine import DefaultTrainer

import os
import cv2
from PIL import Image
import torch
from torchvision import transforms
from fastreid.config import get_cfg
from fastreid.engine import DefaultTrainer
from fastreid.utils.checkpoint import Checkpointer


def load_reid_model():
    cfg_path = '/home/juma/code/StrongSORT/fast-reid/configs/DukeMTMC/bagtricks_S50.yml'
    model_weights = 'checkpoints/FastReID/market_bot_R50.pth'

    cfg = get_cfg()
    cfg.merge_from_file(cfg_path)
    cfg.MODEL.BACKBONE.PRETRAIN = False
    cfg.MODEL.WEIGHTS = model_weights
    model = DefaultTrainer.build_model(cfg)
    model.eval()
    Checkpointer(model).load(cfg.MODEL.WEIGHTS)
    return model


def get_detections_and_features(image_path, yolo_model, reid_model):
    image = cv2.imread(image_path)

    # Get detections
    detections = yolo_model.predict(
        image, classes=[0], verbose=False, imgsz=1280)
    boxes = detections[0].boxes.data.cpu().numpy()

    # Extract re-ID features
    img_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    appearance_features = extract_reid_features(reid_model, img_pil, boxes)

    return boxes, appearance_features


def extract_reid_features(reid_model, img, boxes):
    transform = transforms.Compose([
        transforms.Resize((256, 128)),
        transforms.ToTensor()
    ])

    features_list = []
    for box in boxes:
        x1, y1, x2, y2 = box[:4]
        crop = img.crop((x1, y1, x2, y2))
        tensor_crop = transform(crop).unsqueeze(0).cuda()
        feature = reid_model(tensor_crop).detach().cpu().numpy().squeeze()
        features_list.append(feature)

    return features_list


if __name__ == '__main__':
    image_path = "/home/juma/code/StrongSORT/datasets/MOT17/test/MOT17-01-DPM/img1/000001.jpg"

    yolo_model = YOLO("yolov8m.pt")
    reid_model = load_reid_model()

    boxes, features = get_detections_and_features(
        image_path, yolo_model, reid_model)
    # Now you have detection boxes and corresponding re-ID features.
    # You can proceed with tracking or other tasks.

In [ ]:
features

In [ ]:
features[0].shape


In [ ]:
boxes

In [ ]:
import os
import cv2
from pathlib import Path
import torch
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/home/juma/code/StrongSORT')
from ultralytics import YOLO

def process_frames(folder_path, model):
    
    # Path to the MOT17 sequence frames
    img_folder = os.path.join(folder_path, 'img1')
    img_files = sorted([os.path.join(img_folder, img)
                       for img in os.listdir(img_folder)])

    # Create a results folder if it doesn't exist
    results_folder = os.path.join(folder_path, 'yolov8_detections')
    os.makedirs(results_folder, exist_ok=True)

    # Process each frame
    for img_path in img_files:
        image = cv2.imread(img_path)
        results = model.predict(image,classes=[0])  # Apply detection
        # results[0].boxes.boxes # contains standard yolo boxes as tensor([[1.3376e+03, 4.1875e+02, 1.4995e+03, 7.8562e+02, 8.4918e-01, 0.0000e+00],
        # results[0].appearance_features.shape  is torch.Size([7, 16])
        # complete the code to construct data matrix where each row is detection with 138 columns, first 10 are MOT style detections, next 128 are appearance features
        
        
# Load a model
model = YOLO("yolov8n.pt")  # Load a pretrained model (recommended for training)
# Path to the MOT17 dataset sequence
folder_path = '/home/juma/code/StrongSORT/datasets/MOT17/train/MOT17-02-FRCNN'
frame_results = process_frames(folder_path, model)
frame_results

In [ ]:
from ultralytics import YOLO
import os
import cv2
from pathlib import Path
import torch
# %load_ext autoreload
# %autoreload 2
import sys
sys.path.append('/home/juma/code/StrongSORT')


def process_frames(folder_path, model):
    data_matrix = []

    # Path to the MOT17 sequence frames
    img_folder = os.path.join(folder_path, 'img1')
    img_files = sorted([os.path.join(img_folder, img)
                       for img in os.listdir(img_folder)])

    # Create a results folder if it doesn't exist
    results_folder = os.path.join(folder_path, 'yolov8_detections')
    os.makedirs(results_folder, exist_ok=True)

    # Process each frame
    for frame_number, img_path in enumerate(img_files, start=1):
        image = cv2.imread(img_path)
        # Apply detection for 'person' class
        results = model.predict(image, classes=[0])

        boxes = results[0].boxes.boxes.cpu().numpy()
        appearance_features = results[0].appearance_features.cpu().numpy()

        for box, feature in zip(boxes, appearance_features):
            # Constructing the MOT format
            left, top, right, bottom, conf, _ = box
            width = right - left
            height = bottom - top
            mot_format = [frame_number, -1, left,
                          top, width, height, conf, -1, -1, -1]

            # Combining MOT format with appearance features
            detection_data = mot_format + feature.tolist()
            data_matrix.append(detection_data)

    return data_matrix


# Load a model
# Load a pretrained model (recommended for training)
model = YOLO("yolov8n.pt")
# Path to the MOT17 dataset sequence
folder_path = '/home/juma/code/StrongSORT/datasets/MOT17/train/MOT17-02-FRCNN'
frame_results = process_frames(folder_path, model)
print(frame_results)

In [ ]:
frame_results

In [ ]:
frame_results[0].boxes.shape


# FastREID

In [ ]:
!pip install fastreid


In [ ]:
import torch
from fastreid.config import get_cfg
from fastreid.engine import DefaultTrainer
from fastreid.utils.checkpoint import Checkpointer
from torchvision import transforms
from PIL import Image


def load_reid_model():
    cfg_path = '/home/juma/code/StrongSORT/fast-reid/configs/DukeMTMC/bagtricks_S50.yml'
    model_weights = 'checkpoints/FastReID/market_bot_R50.pth'

    cfg = get_cfg()
    cfg.merge_from_file(cfg_path)
    cfg.MODEL.BACKBONE.PRETRAIN = False
    cfg.MODEL.WEIGHTS = model_weights
    model = DefaultTrainer.build_model(cfg)
    model.eval()
    Checkpointer(model).load(cfg.MODEL.WEIGHTS)
    return model


def get_transform(size=(256, 128)):
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor()
    ])
    return transform


def extract_reid_features(model, img, detections):
    transform = get_transform((256, 128))
    features_list = []

    for det in detections:
        crop = img.crop((det[2], det[3], det[2] + det[4], det[3] + det[5]))
        tensor_crop = transform(crop).unsqueeze(0).cuda()
        feature = model(tensor_crop).detach().cpu().numpy()
        features_list.append(feature)

    return features_list


if __name__ == '__main__':
    # Configuration paths

    # Load model and extract features
    reid_model = load_reid_model()

    # Load your image and bounding boxes here
    img_path = 'sample_image.jpg'
    img = Image.open(img_path)
    # For demonstration purpose, using dummy detections. Replace this with your YOLOv8 detections
    detections = [[0, 0, 50, 50, 150, 150], [0, 0, 100, 100, 250, 250]]

    features = extract_reid_features(reid_model, img, detections)

# use pytorch_deepsort
* https://github.com/ZQPei/deep_sort_pytorch


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, is_downsample=False):
        super(BasicBlock, self).__init__()
        self.is_downsample = is_downsample
        if is_downsample:
            self.conv1 = nn.Conv2d(
                c_in, c_out, 3, stride=2, padding=1, bias=False)
        else:
            self.conv1 = nn.Conv2d(
                c_in, c_out, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.relu = nn.ReLU(True)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        if is_downsample:
            self.downsample = nn.Sequential(
                nn.Conv2d(c_in, c_out, 1, stride=2, bias=False),
                nn.BatchNorm2d(c_out)
            )
        elif c_in != c_out:
            self.downsample = nn.Sequential(
                nn.Conv2d(c_in, c_out, 1, stride=1, bias=False),
                nn.BatchNorm2d(c_out)
            )
            self.is_downsample = True

    def forward(self, x):
        y = self.conv1(x)
        y = self.bn1(y)
        y = self.relu(y)
        y = self.conv2(y)
        y = self.bn2(y)
        if self.is_downsample:
            x = self.downsample(x)
        return F.relu(x.add(y), True)


def make_layers(c_in, c_out, repeat_times, is_downsample=False):
    blocks = []
    for i in range(repeat_times):
        if i == 0:
            blocks += [BasicBlock(c_in, c_out, is_downsample=is_downsample),]
        else:
            blocks += [BasicBlock(c_out, c_out),]
    return nn.Sequential(*blocks)


class Net(nn.Module):
    def __init__(self, num_classes=625, reid=False):
        super(Net, self).__init__()
        # 3 128 64
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ELU(inplace=True),
            nn.Conv2d(32, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ELU(inplace=True),
            nn.MaxPool2d(3, 2, padding=1),
        )
        # 32 64 32
        self.layer1 = make_layers(32, 32, 2, False)
        # 32 64 32
        self.layer2 = make_layers(32, 64, 2, True)
        # 64 32 16
        self.layer3 = make_layers(64, 128, 2, True)
        # 128 16 8
        self.dense = nn.Sequential(
            nn.Dropout(p=0.6),
            nn.Linear(128*16*8, 128),
            nn.BatchNorm1d(128),
            nn.ELU(inplace=True)
        )
        # 256 1 1
        self.reid = reid
        self.batch_norm = nn.BatchNorm1d(128)
        self.classifier = nn.Sequential(
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = x.view(x.size(0), -1)
        if self.reid:
            x = self.dense[0](x)
            x = self.dense[1](x)
            x = x.div(x.norm(p=2, dim=1, keepdim=True))
            return x
        x = self.dense(x)
        # B x 128
        # classifier
        x = self.classifier(x)
        return x

In [ ]:
import torch
import torchvision.transforms as transforms
import numpy as np
import cv2
import logging


from fastreid.config import get_cfg
from fastreid.engine import DefaultTrainer
from fastreid.utils.checkpoint import Checkpointer


class Extractor(object):
    def __init__(self, model_path, use_cuda=True):
        self.net = Net(reid=True)
        self.device = "cuda" if torch.cuda.is_available() and use_cuda else "cpu"
        state_dict = torch.load(model_path, map_location=lambda storage, loc: storage)[
            'net_dict']
        self.net.load_state_dict(state_dict)
        logger = logging.getLogger("root.tracker")
        logger.info("Loading weights from {}... Done!".format(model_path))
        print("Loading weights from {}... Done!".format(model_path))
        self.net.to(self.device)
        self.size = (64, 128)
        self.norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def _preprocess(self, im_crops):
        """
        TODO:
            1. to float with scale from 0 to 1
            2. resize to (64, 128) as Market1501 dataset did
            3. concatenate to a numpy array
            3. to torch Tensor
            4. normalize
        """
        def _resize(im, size):
            return cv2.resize(im.astype(np.float32)/255., size)

        im_batch = torch.cat([self.norm(_resize(im, self.size)).unsqueeze(
            0) for im in im_crops], dim=0).float()
        return im_batch

    def __call__(self, im_crops):
        im_batch = self._preprocess(im_crops)
        with torch.no_grad():
            im_batch = im_batch.to(self.device)
            features = self.net(im_batch)
        return features.cpu().numpy()


img = cv2.imread("sample_image.jpg")

img = img[:, :, (2, 1, 0)]
extr = Extractor(
    "/home/juma/code/StrongSORT/checkpoints/FastReID/deepsort/original_ckpt.t7")


feature = extr([img])
print(feature.shape)

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
from deep_sort.deep.feature_extractor import Extractor


def extract_features(image_path, model_path):
    # Load pre-trained model for feature extraction
    model = Extractor(model_path)

    # Open the image and prepare it for processing
    img = Image.open(image_path).convert('RGB')

    # The repository uses this transform for preprocessing
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[
                             0.229, 0.224, 0.225])
    ])

    # Apply transformations and add batch dimension
    img_tensor = transform(img).unsqueeze(0)

    # Extract appearance features
    with torch.no_grad():
        features = model(img_tensor)

    return features


# Example usage
# Change this to the path of the model checkpoint
model_path = "path_to_deepsort_model_checkpoint.ckpt"
image_path = "example.jpg"
features = extract_features(image_path, model_path)
print(features)

In [ ]:
features[1].shape
